# TalentPulse AI: step-by-step data cleaning (testing_roles_4.xlsx)
Run the cells top to bottom. Every step prints a **before / after** so you can see what changed, and a short **Inference** note explains why.

**How to use in Google Colab:** Colab opens `.ipynb` files. Either upload the `.ipynb` version, or in a new notebook paste each block between `# %%` markers into its own cell.
**Runtime:** about 6 to 8 minutes in total. Step 0 is the slow one (reading a 110 MB Excel file).
**Privacy:** the file contains personal data. Run it only in your own Colab or machine, and do not share outputs publicly.

In [ ]:
# STEP 0a. Setup and file upload
import os, re, io, csv, zipfile, datetime as dt, hashlib
from collections import Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print
pd.set_option('display.max_columns', 40, 'display.width', 200, 'display.max_colwidth', 40)

try:                                             # inside Colab: pick the xlsx from your computer
    from google.colab import files
    up = files.upload()
    XLSX = next(iter(up))
except ImportError:                              # outside Colab: set path here or via environment variable
    XLSX = os.environ.get('XLSX', 'testing_roles_4.xlsx')
# Alternative in Colab if the upload is slow: mount Drive and set XLSX = '/content/drive/MyDrive/testing_roles_4.xlsx'
print('Using file:', XLSX, f'({os.path.getsize(XLSX)/1e6:.1f} MB)')

## Step 0b. Read the workbook the safe way
`pd.read_excel` would try to load a 400 MB sheet into memory and often crashes or takes 20+ minutes. We **stream** the XML instead.
Two things are captured on the way in: (1) every cell as text, so nothing is silently converted, and (2) cells where Excel stored an **error** (`#NAME?`) together with the formula text hidden behind it (used in Step 1).

In [ ]:
from lxml import etree
NS = '{http://schemas.openxmlformats.org/spreadsheetml/2006/main}'
EXPECTED = ['notice_period','Name','Role','Role ID','Skill Cluster','Skill Cluster ID','Experience','Experience Band',
 'Experience Band ID','Salary','Salary Raw','Current Location','Current Location ID','Current Role','Current Company',
 'Previous Role','Previous Company','Education','Preferred Locations','Key Skills','Total Views','Total Downloads',
 'Last Modified','Last Active','Profile Summary','Logs']
FORMULA_TEXT = {}   # (row index, column name) -> original text hidden inside a "formula"

def col_index(ref):
    n = 0
    for ch in re.match(r'[A-Z]+', ref).group(0):
        n = n * 26 + ord(ch) - 64
    return n - 1

def stream_xlsx(path):
    zf = zipfile.ZipFile(path)
    sst = []
    with zf.open('xl/sharedStrings.xml') as f:
        for _, el in etree.iterparse(f, tag=NS + 'si'):
            sst.append(''.join(t.text or '' for t in el.iter(NS + 't')))
            el.clear()
            while el.getprevious() is not None:
                del el.getparent()[0]
    rows = []
    with zf.open('xl/worksheets/sheet1.xml') as f:
        for _, row in etree.iterparse(f, tag=NS + 'row'):
            vals = [''] * 26
            for c in row.iter(NS + 'c'):
                v = c.find(NS + 'v'); t = c.get('t'); i = col_index(c.get('r'))
                if i >= 26:
                    continue
                if t == 'e':                       # Excel error cell, e.g. #NAME?
                    fe = c.find(NS + 'f')
                    if fe is not None and fe.text:
                        FORMULA_TEXT[(len(rows) - 1, EXPECTED[i])] = fe.text
                    vals[i] = v.text if v is not None else ''
                elif t == 'inlineStr':
                    vals[i] = ''.join(x.text or '' for x in c.iter(NS + 't'))
                elif v is None:
                    continue
                elif t == 's':
                    vals[i] = sst[int(v.text)]
                else:
                    vals[i] = v.text
            rows.append(vals)
            row.clear()
            while row.getprevious() is not None:
                del row.getparent()[0]
    return rows

rows = stream_xlsx(XLSX)
header, data = [h.strip() for h in rows[0]], rows[1:]
assert header == EXPECTED, 'Header differs from the expected 26 columns'
df = pd.DataFrame(data, columns=EXPECTED)
df.insert(0, 'source_row', np.arange(2, len(df) + 2))   # Excel row number, kept for traceability
del rows, data
RAW_ROWS = len(df)
print(f'Raw data rows: {RAW_ROWS:,} | columns: {df.shape[1]-1}')
display(df.head(3))

## First look: what is in each column?
**Inference to watch for:** columns that are 100% blank or constant carry no information; columns with very high blank rates cannot be relied on.

In [ ]:
profile = pd.DataFrame({
    'blank_%': (df[EXPECTED].apply(lambda s: (s.str.strip() == '').mean()) * 100).round(2),
    'distinct_values': df[EXPECTED].nunique(),
    'example': df[EXPECTED].iloc[0].str[:30],
})
display(profile)
print('INFERENCE: Role, Role ID, Skill Cluster, Skill Cluster ID are 100% blank (no provider taxonomy exists).')
print('           notice_period has one dominant value (constant) -> no information. Salary Raw duplicates Salary.')

## Step 1. Recover text that Excel turned into errors
When a text cell begins with `-` or `=`, Excel reads it as a formula and shows `#NAME?`. The original text survives inside the formula, so we can restore it. Excel also injects a tag like `_xlnm.Database` where the word "Database" appeared.

In [ ]:
err_mask = df[EXPECTED].apply(lambda s: s.str.startswith('#')).any(axis=1)
print('Cells showing an Excel error before fix:', int(sum(df[c].str.match(r'^#(NAME|REF|VALUE|N/A|DIV)').sum() for c in EXPECTED)))
before = [(k, df.at[k[0], k[1]]) for k in list(FORMULA_TEXT)[:3]]
for (r, c), text in FORMULA_TEXT.items():
    df.at[r, c] = text.replace('_xlnm.', '')
print('Cells recovered from hidden formula text:', len(FORMULA_TEXT))
print(Counter(c for _, c in FORMULA_TEXT))
for (k, old) in before:
    print(f'\nColumn {k[1]}: BEFORE = {old!r}\n               AFTER  = {df.at[k[0], k[1]][:90]!r}')
print('\nINFERENCE: this file was opened and re-saved in Excel. Never edit this data in Excel; always re-run the cleaning on the raw file.')

## Step 2. Find and realign column-shifted rows
**Rule:** column A (`notice_period`) should be a number. If it is text and the next column (`Name`) is blank, the whole row has slid one column to the left.

In [ ]:
is_num = df['notice_period'].str.match(r'^\d+(\.\d+)?$')
shifted = (~is_num) & (df['Name'].str.strip() == '')
print(f'Shifted rows: {shifted.sum():,}  (Excel rows {df.loc[shifted,"source_row"].min():,} to {df.loc[shifted,"source_row"].max():,})')
print('Last column (Logs) blank in all shifted rows (so nothing falls off the edge):', bool((df.loc[shifted, 'Logs'] == '').all()))
show = ['source_row', 'notice_period', 'Name', 'Experience', 'Experience Band', 'Salary', 'Current Location']
print('\nBEFORE (note: names sit under notice_period, salary sits under the band column):')
display(df.loc[shifted, show].head(3))

cols = EXPECTED
df.loc[shifted, cols[1:]] = df.loc[shifted, cols[:-1]].values     # move every value one column to the right
df.loc[shifted, 'notice_period'] = ''                              # that column did not exist in the source batch
df['shift_repaired_flag'] = shifted                                # remember which rows were repaired
print('AFTER:')
display(df.loc[shifted, show].head(3))
print('INFERENCE: this block came from a different export without the notice_period column. Unfixed, salary would be read as experience and 6,448 profiles would be mis-ranked.')

## Step 3. Decode experience bands that Excel turned into dates
A band like `2-4` was converted to the date 4 Feb 2026, stored as the number 46057. We reverse it: number -> date -> `month-day`.

In [ ]:
print('BEFORE:'); display(df['Experience Band'].value_counts().head(10).to_frame())
def decode_band(x):
    x = x.strip()
    if re.fullmatch(r'\d{5}', x):
        d = dt.date(1899, 12, 30) + dt.timedelta(days=int(x))
        return f'{d.month}-{d.day}'
    return x
n_serial = df['Experience Band'].str.fullmatch(r'\d{5}').sum()
df['Experience Band'] = df['Experience Band'].map(decode_band)
print(f'\nDecoded {n_serial:,} values ({n_serial/len(df):.1%} of raw rows; 91.7% once duplicates are removed). AFTER:'); display(df['Experience Band'].value_counts().to_frame())
chk = df.groupby('Experience Band')['Experience Band ID'].nunique().max()
print('Each band maps to exactly one band ID:', chk == 1)
print('INFERENCE: the data passed through Excel with US date settings. Do not open the cleaned CSV in Excel (it will corrupt bands again); use the xlsx where bands are stored as text.')

## Step 4. Remove exact duplicates
Two rows are exact duplicates when every populated field is identical (we ignore `notice_period`, which is blank for the realigned rows). We keep the first occurrence and its original Excel row number.

In [ ]:
key_cols = EXPECTED[1:]
dup_all = df.duplicated(subset=key_cols, keep='first')
print(f'Exact duplicate rows: {dup_all.sum():,} of {len(df):,} ({dup_all.mean():.1%})')
ex = df[dup_all].iloc[0]
same = (df['Name'] == ex['Name']) & (df['Education'] == ex['Education']) & (df['Key Skills'] == ex['Key Skills'])
print('\nExample: the same profile appears this many times (source_row shows where):')
display(df.loc[same, ['source_row', 'Name', 'Experience', 'Current Role', 'Current Company']].head(5))
df = df[~dup_all].reset_index(drop=True)
print(f'Rows after de-duplication: {len(df):,}')
print('INFERENCE: one row in three was a repeat. Left in, one candidate could take several top-10 slots and inflate ranking metrics.')

## Step 5. Flag near-duplicates (flag only, do not delete)
Same name + education + current company as another row, but some other field differs (for example a refreshed skills list).

In [ ]:
nk = df['Name'].str.lower().str.strip() + '|' + df['Education'].str.lower().str.strip() + '|' + df['Current Company'].str.lower().str.strip()
df['near_duplicate_flag'] = nk.duplicated(keep=False)
print(f"Rows flagged: {df['near_duplicate_flag'].sum():,} ({df['near_duplicate_flag'].mean():.1%})")
display(df[df['near_duplicate_flag']].sort_values('Name')[['Name', 'Experience', 'Current Role', 'Current Company', 'Total Views']].head(6))
print('INFERENCE: probably repeat snapshots of the same person, but blank company fields can collide, so treat 33.7% as an upper bound.')
print('           Use this flag so one person never lands in both the training and test sets.')

## Step 6. Experience: convert to numbers and sanity-check

In [ ]:
df['experience_years'] = pd.to_numeric(df['Experience'], errors='coerce').round(2)
print('Not numeric:', df['experience_years'].isna().sum(), '| negative:', (df['experience_years'] < 0).sum())
display(df['experience_years'].describe().round(2).to_frame().T)
print(f"At the 12.0 ceiling: {(df['experience_years']==12).mean():.1%} | at the 1.0 floor: {(df['experience_years']==1).mean():.1%}")
bands = df.groupby('Experience Band')['experience_years'].agg(['min', 'max', 'count']); display(bands)
df['experience_years'].plot.hist(bins=48, figsize=(7, 2.6), color='#1F3A5F', title='Experience (years)'); plt.show()
print('INFERENCE: values run only from 1 to 12 and the 12-15 band is all exactly 12.0, so the pool was filtered and top-coded by the provider.')
print('           No outliers to delete, but freshers and 12+ year profiles cannot be studied with this data.')

## Step 7. Salary: zeros, plausibility flags, outliers

In [ ]:
sal = pd.to_numeric(df['Salary'], errors='coerce')
df['salary_zero_flag'] = sal == 0
df['salary_inr'] = sal.where(sal > 0)                         # zero means "not disclosed", so set to blank
df['salary_implausible_flag'] = (df['salary_inr'] > 1e7) | (df['salary_inr'] < 1e5)
print(f"Zero (not disclosed): {df['salary_zero_flag'].sum():,} ({df['salary_zero_flag'].mean():.1%})")
print(f"Below Rs 1 lakh: {(df['salary_inr']<1e5).sum():,} | Above Rs 1 crore: {(df['salary_inr']>1e7).sum():,}  -> flagged, values NOT edited")
display((df['salary_inr'] / 1e5).describe(percentiles=[.01, .5, .99]).round(2).to_frame('salary (Rs lakh)').T)
np.log10(df['salary_inr'].dropna()).plot.hist(bins=60, figsize=(7, 2.6), color='#1B998B', title='log10(Salary in Rs)'); plt.show()
print('INFERENCE: very low values look like monthly pay typed as annual; the long right tail is a few extreme entries.')
print('           Salary is sensitive and uneven across seniority and city, so it stays out of ranking.')

## Step 8. Skills: split, trim, remove repeats inside each profile

In [ ]:
def clean_skills(s):
    seen, out = set(), []
    for t in re.split(r'[,;|]', s or ''):
        t = re.sub(r'\s+', ' ', t).strip()
        if t and t.lower() not in seen:
            seen.add(t.lower()); out.append(t)
    return out
skills = df['Key Skills'].map(clean_skills)
raw_tokens = df['Key Skills'].map(lambda s: len([t for t in re.split(r'[,;|]', s or '') if t.strip()]))
df['n_skills'] = skills.map(len)
df['key_skills'] = skills.map('|'.join)
print('Duplicate skill tokens removed:', int((raw_tokens - df['n_skills']).sum()))
display(pd.DataFrame({'BEFORE': df['Key Skills'].str[:70].head(3), 'AFTER': df['key_skills'].str[:70].head(3)}))
top = Counter(s.lower() for l in skills for s in l).most_common(12)
pd.Series(dict(top)).iloc[::-1].plot.barh(figsize=(7, 3), color='#F2A541', title='Most common skills (profiles)'); plt.show()
print(f"Median skills per profile: {df['n_skills'].median():.0f} | profiles with zero skills: {(df['n_skills']==0).sum()}")
print('INFERENCE: skills are the strongest field. Synonyms (Selenium vs Selenium Webdriver) are still separate: a synonym map is the next improvement.')

## Step 9. Locations, role family and education

In [ ]:
ALIAS = {'bangalore':'Bengaluru','bengaluru':'Bengaluru','bangaluru':'Bengaluru','gurgaon':'Gurugram','gurugram':'Gurugram','delhi':'New Delhi','new delhi':'New Delhi',
         'bombay':'Mumbai','madras':'Chennai','calcutta':'Kolkata','trivandrum':'Thiruvananthapuram','cochin':'Kochi','vizag':'Visakhapatnam','mysore':'Mysuru','poona':'Pune','secunderabad':'Hyderabad'}
def city_std(loc):
    first = re.sub(r'[^a-z ]', '', re.split(r'[,(/.]', loc.strip())[0].lower()).strip()
    return ALIAS.get(first, first.title()) if first else ''
df['current_city_std'] = df['Current Location'].map(city_std)
print(f"Raw location labels: {df['Current Location'].nunique():,} -> standard cities: {df['current_city_std'].nunique():,}")
c = df['current_city_std'].value_counts(); print(f'Top 10 cities hold {c.head(10).sum()/len(df):.1%} of profiles')
c.head(10).iloc[::-1].plot.barh(figsize=(7, 3), color='#1F3A5F', title='Profiles by standard city'); plt.show()

ROLE_RULES = [('QA / Testing', r'\btest|\bqa\b|quality|sdet|\bqe\b|automation|selenium'),
 ('ETL / Data Engineering', r'\betl\b|data ?ware|dwh|ab ?initio|informatica|datastage|big data|data engineer|bi developer'),
 ('DevOps / Cloud / Infra', r'devops|\bcloud\b|\bsre\b|infrastructure|system admin|sysadmin|network'),
 ('Data / Analytics', r'analyst|analytics|data scien|business intell|\bbi\b'),
 ('Management / Lead', r'manager|\blead\b|head|director|architect'),
 ('Software Development', r'develop|software|programmer|\bsde\b|full ?stack|engineer|java|python|\.net'),
 ('Support / Operations', r'support|operations|process|associate|executive|consultant')]
RES = [(n, re.compile(p, re.I)) for n, p in ROLE_RULES]
def role_family(cur, prev, sk):
    for src, txt in (('current_role', cur), ('previous_role', prev)):
        if txt:
            for n, rx in RES:
                if rx.search(txt): return n, src
            return 'Other', src
    for n, rx in RES[:2]:
        if sk and rx.search(sk): return n, 'key_skills'
    return 'Unclassified (no role text)', 'none'
rf = [role_family(a, b, k) for a, b, k in zip(df['Current Role'], df['Previous Role'], df['Key Skills'])]
df['role_family'], df['role_family_source'] = [x[0] for x in rf], [x[1] for x in rf]
display(df['role_family'].value_counts().to_frame())
print('INFERENCE: with the provider taxonomy empty, role_family is our rule-based approximation. QA/testing dominates, so results will not transfer to other occupations.')

EDU = [('Bachelor - Engineering/Technology', r'^(b\.?\s?tech|b\.?\s?e\b|bachelor of engineering|be/)'), ('Master - Engineering/Technology', r'^(m\.?\s?tech|m\.?\s?e\b|master of (engineering|technology))'),
 ('Master - Computer Applications', r'^(mca|master of computer)'), ('Master - Management', r'^(mba|pgdm|post graduate diploma in management)'),
 ('Master - Other', r'^(ms\b|m\.?sc|ms/m\.?sc|m\.?a\b|m\.?com|master)'), ('Bachelor - Other', r'^(b\.?\s?sc|bca|b\.?\s?com|b\.?\s?a\b|bba|bachelor)'),
 ('PG Diploma / Diploma', r'^(pg diploma|post graduate diploma|diploma)'), ('Doctorate', r'^(ph\.?d|doctorate)'), ('School (10th/12th)', r'^(12th|10th|class)'), ('Not pursuing / not stated', r'^(not pursuing)')]
EDURES = [(n, re.compile(p, re.I)) for n, p in EDU]
def edu_level(e):
    if not e.strip(): return ''
    for n, rx in EDURES:
        if rx.search(e.strip()): return n
    return 'Other / unparsed'
df['education_level'] = df['Education'].map(edu_level)
def grad_year(e):
    ys = re.findall(r'(?:19|20)\d\d', e)
    return int(ys[-1]) if ys and 1990 <= int(ys[-1]) <= 2031 else pd.NA
df['grad_year'] = df['Education'].map(grad_year).astype('Int64')
display(df[['Education', 'education_level', 'grad_year']].head(4))
more = df['Preferred Locations'].str.extract(r'\+\s?(\d+)\s+more')[0].astype(float).fillna(0).astype(int)
df['pref_locations_truncated'] = more
df['preferred_locations'] = df['Preferred Locations'].str.replace(r'\+\s?\d+\s+more', '', regex=True).str.replace(r'\s*,\s*', '|', regex=True).str.strip(' |,')
print(f'Preferred-location lists cut off by "+N more": {(more>0).sum():,} rows')
print('INFERENCE: institution names are dropped (prestige can proxy background); degree level and graduation year are kept. Truncated lists understate how mobile a candidate is.')

## Step 10. Engagement and consistency flags

In [ ]:
views = pd.to_numeric(df['Total Views'], errors='coerce'); dls = pd.to_numeric(df['Total Downloads'], errors='coerce')
df['total_views'], df['total_downloads'] = views, dls
df['downloads_gt_views_flag'] = dls > views
df['has_profile_summary'] = df['Profile Summary'].str.strip() != ''
df['has_logs'] = df['Logs'].str.strip() != ''
print(f"Rows with downloads > views (inconsistent): {df['downloads_gt_views_flag'].sum():,}")
display(views.describe(percentiles=[.5, .99]).round(1).to_frame('views').T)
print(f"Profile summary present: {df['has_profile_summary'].mean():.1%} | Logs filled: {df['has_logs'].mean():.1%}")
print('Recency labels:'); display(df['Last Active'].value_counts().head(6).to_frame())
print('INFERENCE: views are heavy-tailed and reflect platform exposure, not quality; recency labels are relative to the extraction date. All stay descriptive and out of scoring.')

## Step 11. Drop identifying and useless fields, pseudonymise, order columns

In [ ]:
df = df.rename(columns={'Experience Band': 'experience_band', 'Experience Band ID': 'experience_band_id', 'Current Location': 'current_location',
    'Current Location ID': 'current_location_id', 'Current Role': 'current_role', 'Previous Role': 'previous_role',
    'Last Modified': 'last_modified_label', 'Last Active': 'last_active_label'})
df['experience_band_id'] = pd.to_numeric(df['experience_band_id'], errors='coerce')
df['record_id'] = ['TP%06d' % i for i in range(1, len(df) + 1)]
FINAL = ['record_id','source_row','experience_years','experience_band','experience_band_id','salary_inr','salary_zero_flag','salary_implausible_flag',
 'current_location','current_city_std','current_location_id','current_role','previous_role','role_family','role_family_source','education_level','grad_year',
 'preferred_locations','pref_locations_truncated','key_skills','n_skills','total_views','total_downloads','downloads_gt_views_flag',
 'last_modified_label','last_active_label','has_profile_summary','has_logs','shift_repaired_flag','near_duplicate_flag']
dropped = [c for c in EXPECTED if c not in ('Experience Band','Experience Band ID','Current Location','Current Location ID','Current Role','Previous Role','Last Modified','Last Active')]
print('Dropped / replaced source columns:', dropped)
clean = df[FINAL].copy()
print(f'Clean shape: {clean.shape}'); display(clean.head(3))
print('INFERENCE: names, employers, summaries and logs are removed to minimise personal data. The file is still personal data (role + city + experience + skills can identify someone), so keep it private.')

## Step 12. Final checks and save

In [ ]:
print('Rows: raw', f'{RAW_ROWS:,}', '-> clean', f'{len(clean):,}')
print('record_id unique:', clean['record_id'].is_unique)
miss = (clean.replace('', np.nan).isna().mean() * 100).round(2).sort_values(ascending=False)
display(miss[miss > 0].to_frame('blank %'))
print('Rule: blanks are NOT filled. "Not stated" is not "does not meet"; imputing a role or skill would invent a qualification.')
for b in clean.select_dtypes('bool'): clean[b] = clean[b].map({True: 'true', False: 'false'})
clean.to_csv('candidate_profiles_clean.csv.gz', index=False, compression='gzip')
print('Saved candidate_profiles_clean.csv.gz', f'({os.path.getsize("candidate_profiles_clean.csv.gz")/1e6:.1f} MB)')
try:
    files.download('candidate_profiles_clean.csv.gz')
except NameError:
    pass

## Summary: what changed and what it means

In [ ]:
summary = pd.DataFrame([
 ('Rows', f'{RAW_ROWS:,}', f'{len(clean):,}', 'exact duplicates removed'),
 ('Excel error cells (#NAME?)', f'{len(FORMULA_TEXT)}', '0', 'text recovered from hidden formulas'),
 ('Column-shifted rows', f'{int(shifted.sum()):,}', '0', 'realigned by rule'),
 ('Experience bands as date numbers', f'{n_serial:,}', '0', 'decoded back to labels'),
 ('Columns', '26', f'{clean.shape[1]}', '11 dropped, 15 derived'),
 ('Salary 0 (undisclosed)', f"{int(df['salary_zero_flag'].sum()):,}", 'blank', 'never imputed'),
], columns=['Item', 'Before', 'After', 'Action'])
display(summary)
print('\nKey takeaways:\n 1. The file is clean enough for the retrieval study, but only QA/testing talent, 1-12 years, metros.\n 2. Rank on role, skills and experience; keep salary, views, recency and location preference out of scoring.\n 3. Always re-run this notebook on the raw file instead of editing in Excel.')